In [ ]:
# -- Cell 1 -- rclone + Drive, same pattern as the CellPPD notebook.# Kaggle is a fresh machine each session. Requires: Settings -> Internet ON and# the RCLONE_DRIVE_TOKEN secret attached to THIS notebook.import os, subprocessr = subprocess.run("curl -s https://rclone.org/install.sh | sudo bash", shell=True)if r.returncode not in (0, 3):          # 3 = already installed and current    raise RuntimeError("rclone install failed (exit %d)" % r.returncode)from kaggle_secrets import UserSecretsClienttoken = UserSecretsClient().get_secret("RCLONE_DRIVE_TOKEN")os.makedirs("/root/.config/rclone", exist_ok=True)with open("/root/.config/rclone/rclone.conf", "w") as f:    f.write("[drive]\ntype = drive\nscope = drive\ntoken = " + token + "\n")REMOTE = "drive:Distillation"out = subprocess.run("rclone lsf " + REMOTE, shell=True, capture_output=True, text=True)print(out.stdout or out.stderr)assert out.returncode == 0, "cannot see " + REMOTE

In [ ]:
# -- Cell 2 -- deps. pyarrow + huggingface_hub give us HTTP range reads into the# parquet files, so we fetch only the row groups we want instead of all 34 GB.subprocess.run('pip install -q -U "huggingface_hub>=0.34" pyarrow', shell=True, check=True)import pyarrow as pa, pyarrow.parquet as pq, pandas as pd, numpy as npfrom huggingface_hub import HfFileSystemprint("pyarrow", pa.__version__, "| pandas", pd.__version__)# The k-mer tokenizer is needed for the length filter. It lives with the model on# Drive; pulling one snapshot's small files is faster than the whole models dir.TOKDIR = "/kaggle/working/tokenizer"if not os.path.exists(TOKDIR + "/tokenizer.json"):    snaps = "%s/models/models--aaronfeller--peptideclm-2-mlm-large/snapshots" % REMOTE    sha = subprocess.run("rclone lsf " + snaps, shell=True, capture_output=True,                         text=True).stdout.split()[0].rstrip("/")    os.makedirs(TOKDIR, exist_ok=True)    subprocess.run('rclone copy %s/%s %s --include "*.json" --include "*.py" -P'                   % (snaps, sha, TOKDIR), shell=True, check=True)subprocess.run('pip install -q -U "transformers>=5.0"', shell=True, check=True)from transformers import AutoTokenizertok = AutoTokenizer.from_pretrained(TOKDIR, trust_remote_code=True)print("tokenizer vocab:", tok.vocab_size)

In [ ]:
# -- Cell 3 -- open the remote parquet files and state the ground truth about them.## THE FILENAMES ARE SWAPPED. Verified against the `source` column and the row# counts in the paper:#     train/peptides.parquet        -> 108,116,047 rows, source PubChem_clean4#     train/small_molecules.parquet ->   9,538,596 rows, source ESMAtlas_clust30#     train/lipids.parquet          ->      48,654 rows, source LMSD_clean   (ok)# Building the subset off the filenames would invert the composition entirely, so# every reference below goes through this table, and Cell 4 re-verifies `source`.BASE = "datasets/aaronfeller/peptideclm-2-pretraining-data"fs = HfFileSystem()FILES = {                     # logical name -> (path, expected source prefix)    "small_molecules": ("train/peptides.parquet",        "PubChem"),    "peptides":        ("train/small_molecules.parquet", "ESMAtlas"),    "lipids":          ("train/lipids.parquet",          "LMSD"),}HANDLES, META = {}, {}for name, (path, _) in FILES.items():    p = pq.ParquetFile(fs.open(BASE + "/" + path))    HANDLES[name] = p    META[name] = (p.metadata.num_rows, p.metadata.num_row_groups)    print("%-16s <- %-32s rows {:,} groups {}".format(*META[name])          % (name, path))ALL_COLS = HANDLES["lipids"].schema_arrow.namesDESC_COLS = [c for c in ALL_COLS             if c not in ("source", "split", "smiles", "__index_level_0__")]print("\n%d descriptor columns (these are the MTR targets)" % len(DESC_COLS))assert len(DESC_COLS) == 99, "expected the 99 RDKit descriptors, got %d" % len(DESC_COLS)

In [ ]:
# -- Cell 4 -- sample row groups.## WHY CLUSTER SAMPLING: parquet's unit of access is the row group. Drawing 1M# uniformly random rows out of 108M would touch nearly all 1,362 groups, i.e.# download the entire 28 GB file. Instead we read a STRIDE of groups spread across# the file and subsample within them: ~1.5 GB fetched instead of 34 GB.## WHY THAT IS SAFE HERE: the file is effectively pre-shuffled. KS tests between the# first and last row group gave D = 0.012-0.018, p = 0.39-0.84 on MolWt, MolLogP and# TPSA -- statistically indistinguishable. Position in the file carries no signal.# Cell 6 re-checks this on the sample actually drawn.TARGET = {"small_molecules": 1_006_000, "peptides": 969_000, "lipids": 48_654}OVERDRAW = 1.25          # extra rows so filters cannot starve the targetKEEP = ["source", "smiles"] + DESC_COLSrng = np.random.default_rng(0)raw = {}for name in ["small_molecules", "peptides", "lipids"]:    p, (nrows, ngroups) = HANDLES[name], META[name]    per_group = nrows / ngroups    need = min(TARGET[name] * OVERDRAW, nrows)    n_groups = int(min(ngroups, np.ceil(need / per_group)))    stride = max(1, ngroups // n_groups)    picks = list(range(0, ngroups, stride))[:n_groups]    if len(picks) == ngroups:        # Taking every group anyway (lipids): one sequential read beats 1,362        # separate range requests, which would be latency-bound on a 50 MB file.        df = p.read(columns=KEEP).to_pandas()        print("  %-16s whole file in one read" % name)    else:        parts = []        for i, g in enumerate(picks):            parts.append(p.read_row_group(g, columns=KEEP).to_pandas())            if (i + 1) % 25 == 0 or i + 1 == len(picks):                print("  %-16s %3d/%3d groups  %s rows"                      % (name, i + 1, len(picks), "{:,}".format(sum(map(len, parts)))))        df = pd.concat(parts, ignore_index=True)    raw[name] = df    print("%-16s %s rows from %d/%d groups (stride %d)\n"          % (name, "{:,}".format(len(df)), len(picks), ngroups, stride))# Re-verify the swapped-filename mapping on the data actually pulled.for name, (_, expect) in FILES.items():    got = raw[name].source.iloc[0]    assert got.startswith(expect), "%s: expected %s, got %s" % (name, expect, got)    print("verified %-16s source = %s" % (name, got))

In [ ]:
# -- Cell 5 -- filter, dedup globally, then draw exact counts.## The upstream files are already cleaned (clean4 / clust30 / clean) and I measured# zero duplicate SMILES and zero NaN/inf descriptors within a row group, so the# per-source pass is light. Two filters bite:##   1. LENGTH <= 512 k-mer tokens. Attention is quadratic and peptides reach 592#      tokens; the longest ~4.5% cost ~34% more attention for no benefit. PubChem#      is untouched (max 179).#   2. CROSS-SOURCE DEDUP. This is the one that matters and the first version of#      this cell got wrong: deduping per source is not enough, because the same#      molecule appears in more than one source -- LMSD lipids are drug-like and#      also sit in PubChem. Pooling three individually-unique frames therefore#      still yields duplicates. We dedup the POOL, letting the rarest source win#      (lipids > peptides > small_molecules) so the scarce chemistry is never the#      copy that gets dropped.## NO benchmark-overlap removal, by decision: the teacher was trained on the full# corpus, so leaving overlaps in keeps student-vs-teacher fair. Only absolute# benchmark scores are inflated, and equally for both.MAXLEN = 512BATCH = 20000PRIORITY = {"lipids": 0, "peptides": 1, "small_molecules": 2}   # lower winsdef token_lengths(smiles):    out = np.empty(len(smiles), dtype=np.int32)    for i in range(0, len(smiles), BATCH):        chunk = smiles[i:i + BATCH]        out[i:i + len(chunk)] = [len(x) for x in tok(list(chunk))["input_ids"]]    return outclean = {}for name, df in raw.items():    n0 = len(df)    df = df.drop_duplicates(subset="smiles")    n1 = len(df)    df = df.assign(n_tokens=token_lengths(df.smiles.values))    df = df[df.n_tokens <= MAXLEN]    clean[name] = df.assign(_src=name)    print("%-16s %s -> dedup %s -> len<=%d %s"          % (name, "{:,}".format(n0), "{:,}".format(n1), MAXLEN, "{:,}".format(len(df))))pool = pd.concat(clean.values(), ignore_index=True)before = len(pool)pool = (pool.assign(_prio=pool._src.map(PRIORITY))            .sort_values("_prio", kind="stable")            .drop_duplicates(subset="smiles", keep="first"))print("\ncross-source duplicates removed: %s" % "{:,}".format(before - len(pool)))print("pool after global dedup: %s" % "{:,}".format(len(pool)))kept = {}for name in ["small_molecules", "peptides", "lipids"]:    d = pool[pool._src == name]    want = TARGET[name]    if len(d) > want:        d = d.sample(want, random_state=0)    elif len(d) < want:        print("  WARNING %s: only %s rows available, wanted %s"              % (name, "{:,}".format(len(d)), "{:,}".format(want)))    kept[name] = d    print("%-16s drawn %s" % (name, "{:,}".format(len(d))))subset = (pd.concat(kept.values(), ignore_index=True)            .drop(columns=["_src", "_prio"])            .sample(frac=1, random_state=0)            .reset_index(drop=True))print("\nTOTAL %s molecules" % "{:,}".format(len(subset)))assert subset.smiles.duplicated().sum() == 0, "dedup failed"

In [ ]:
# -- Cell 6 -- sanity checks before writing anything.## The descriptors ship PRE-NORMALIZED (z-scored against global corpus statistics),# NOT raw: stored MolWt is -0.43 for PubChem and +1.85 for peptides, where raw# values would be ~350 and ~1200. Two consequences, both easy to get wrong:#   - do NOT normalize again (it would rescale the MTR targets away from what the#     teacher was trained against)#   - do NOT recompute with RDKit (raw values, different scale, silently broken loss)# What they DO need is outlier clipping: the max across the 99 columns is ~452.print("composition:")for s, n in subset.source.value_counts().items():    print("   %-32s %9s  %5.1f%%" % (s, "{:,}".format(n), 100 * n / len(subset)))print("\ntoken lengths by source:")for s, g in subset.groupby("source"):    print("   %-32s median %4d  p95 %4d  max %4d"          % (s, g.n_tokens.median(), np.percentile(g.n_tokens, 95), g.n_tokens.max()))print("   %-32s mean %.1f  -> %s tokens total"      % ("ALL", subset.n_tokens.mean(), "{:,}".format(int(subset.n_tokens.sum()))))D = subset[DESC_COLS].to_numpy(dtype=np.float32)print("\ndescriptors: mean %.3f std %.3f min %.2f max %.2f | NaN %d inf %d"      % (D.mean(), D.std(), D.min(), D.max(), np.isnan(D).sum(), np.isinf(D).sum()))CLIP = 10.0frac = float((np.abs(D) > CLIP).mean())print("cells beyond +/-%.0f: %.4f%%  (clip at load time, not here -- keep the data raw)"      % (CLIP, 100 * frac))assert subset.smiles.duplicated().sum() == 0, "duplicates survived"assert not np.isnan(D).any(), "NaN descriptors"print("\nchecks passed")

In [ ]:
# -- Cell 7 -- write parquet + a manifest describing exactly how it was built.# The manifest matters: six months from now the only way to know whether a result# came from this subset is to have recorded the sampling decisions next to it.import json, hashlibOUTDIR = "/kaggle/working/subset"os.makedirs(OUTDIR, exist_ok=True)PARQ = OUTDIR + "/pretrain_subset_2M.parquet"subset.to_parquet(PARQ, index=False, compression="zstd")size_gb = os.path.getsize(PARQ) / 1e9manifest = {    "created_utc": pd.Timestamp.utcnow().isoformat(),    "source_dataset": "aaronfeller/peptideclm-2-pretraining-data",    "filename_note": "upstream train/peptides.parquet holds PubChem and "                     "train/small_molecules.parquet holds ESMAtlas -- names are swapped",    "n_molecules": int(len(subset)),    "composition": {k: int(v) for k, v in subset.source.value_counts().items()},    "targets": TARGET,    "sampling": "row-group cluster sampling at a stride across each file, then "                "uniform subsample; justified by KS D=0.012-0.018 between first "                "and last row group",    "filters": {"max_kmer_tokens": MAXLEN, "dedup": "exact SMILES",                "benchmark_overlap_removed": False},    "descriptors": {"n": len(DESC_COLS), "pre_normalized": True,                    "recommended_clip": CLIP, "columns": DESC_COLS},    "token_stats": {"mean": float(subset.n_tokens.mean()),                    "median": float(subset.n_tokens.median()),                    "p95": float(np.percentile(subset.n_tokens, 95)),                    "total": int(subset.n_tokens.sum())},    "parquet_sha256": hashlib.sha256(open(PARQ, "rb").read(1 << 20)).hexdigest(),    "parquet_gb": round(size_gb, 3),}json.dump(manifest, open(OUTDIR + "/manifest.json", "w"), indent=2)print("wrote %s (%.2f GB)" % (PARQ, size_gb))print(json.dumps({k: v for k, v in manifest.items() if k != "descriptors"}, indent=2))

In [ ]:
# -- Cell 8 -- ship it to Drive, and read the file back to prove it is intact.## Drive is the backup. The primary consumer should be a KAGGLE DATASET made from# this notebook's output (Save Version -> output -> Create Dataset): every later# distillation run then mounts it at /kaggle/input instantly, instead of pulling# ~1.5 GB over rclone each session.back = pd.read_parquet(PARQ)assert len(back) == len(subset) and back.smiles.duplicated().sum() == 0print("re-read OK: %s rows, %d cols" % ("{:,}".format(len(back)), back.shape[1]))DEST = REMOTE + "/data/pretrain_subset_2M"subprocess.run("rclone copy %s %s --drive-chunk-size 64M -P" % (OUTDIR, DEST),               shell=True, check=True)print("\nuploaded to " + DEST)print(subprocess.run("rclone lsf -l " + DEST, shell=True,                     capture_output=True, text=True).stdout)print("NEXT: Save Version -> Output -> Create Dataset, so training runs mount this "      "at /kaggle/input instead of downloading it.")